# Building a Pipeline

You have built extractors, transformers, and loaders. Now wire them together into a pipeline that runs reliably, every day, without you babysitting it.

A pipeline is just functions calling functions. But the difference between a pipeline that works once on your laptop and one that runs in production for months without incident comes down to three things: **clear contracts**, **proper logging**, and **sensible error handling**.

In [ ]:
%pip install -q sqlalchemy

In [ ]:
import pandas as pd
import json
import glob
import os
import logging
from datetime import datetime
from sqlalchemy import create_engine, text

## The pipeline contract

Every ETL pipeline is built from three kinds of function:

```python
def extract(config)            -> pd.DataFrame
def transform(df)              -> pd.DataFrame
def load(df, config)           -> None
```

That is the contract. Each function has a clear signature:

- **Extract** takes configuration (connection strings, file paths, bookmarks) and returns a DataFrame.
- **Transform** takes a DataFrame and returns a new DataFrame. No config, no side effects.
- **Load** takes a DataFrame and configuration, and writes it somewhere. Returns nothing.

The pipeline itself is just composition:

```python
load(transform(extract(config)), config)
```

Why is this contract so valuable?

1. **Each function is independently testable.** You can test `transform` with a hand-crafted DataFrame. No database needed.
2. **Each function is independently replaceable.** Swapping from SQLite to PostgreSQL? Only `extract` and `load` change. `transform` does not care.
3. **The data flow is visible.** You can inspect the DataFrame between stages. You can log its shape, its schema, sample rows.

Let's build a complete pipeline for the Bazaar seller data.

## Configuration

Hard-coded paths and connection strings scattered through your code are a maintenance nightmare. Use a configuration dict to parameterise the pipeline.

In [ ]:
CONFIG = {
    # Source
    "source_db": "sqlite:///../data/bazaar_orders.sqlite",
    "seller_files_dir": "../data/daily_seller_files",
    "api_file": "../data/marketplace_api.json",

    # Destination
    "dest_db": "sqlite:///:memory:",  # In-memory for demo; use a file path in production

    # Transform settings
    "exchange_rates": {
        "GBP": 1.0,
        "EUR": 0.86,
        "USD": 0.79,
    },

    # Bookmarks (in production, these would be loaded from a state file or metadata table)
    "last_order_id": 0,
    "last_seller_file_date": None,
}

In production, this config might come from a YAML file, environment variables, or a config service. The point is that the pipeline code never contains literal paths or credentials.

## Logging

When a pipeline runs at 3am and something goes wrong, logs are the only thing standing between you and a very bad morning. Python's `logging` module is simple and effective.

In [ ]:
# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)

logger = logging.getLogger("bazaar_pipeline")
logger.info("Pipeline logger initialised")

Log at every stage boundary:

- What we are about to do
- How many rows we got
- How long it took (optional but useful)
- Any warnings (missing data, unexpected values)

## The Extract stage

In [ ]:
def extract_orders(config):
    """
    Extract orders from the source database.
    Uses a high-water mark on order_id for incremental extraction.
    Returns (DataFrame, new_bookmark).
    """
    logger.info("Extracting orders (bookmark: order_id > %d)", config["last_order_id"])

    engine = create_engine(config["source_db"])
    query = f"""
        SELECT * FROM orders
        WHERE order_id > {config['last_order_id']}
        ORDER BY order_id
    """

    with engine.connect() as conn:
        df = pd.read_sql(query, conn)

    new_bookmark = int(df["order_id"].max()) if len(df) > 0 else config["last_order_id"]
    logger.info("Extracted %d orders (new bookmark: %d)", len(df), new_bookmark)

    return df, new_bookmark

In [ ]:
def extract_seller_files(config):
    """
    Extract seller CSV files.
    Uses a date-based bookmark to skip already-processed files.
    Returns (DataFrame, new_bookmark).
    """
    data_dir = config["seller_files_dir"]
    last_date = config["last_seller_file_date"]
    logger.info("Extracting seller files from %s (bookmark: %s)", data_dir, last_date)

    csv_files = sorted(glob.glob(os.path.join(data_dir, "*.csv")))
    frames = []

    for filepath in csv_files:
        basename = os.path.basename(filepath)
        parts = basename.replace(".csv", "").split("_")
        file_date = parts[2]

        if last_date and file_date <= last_date:
            continue

        df = pd.read_csv(filepath)
        df["seller_name"] = parts[1]
        df["file_date"] = file_date
        frames.append(df)

    if frames:
        result = pd.concat(frames, ignore_index=True)
        new_bookmark = result["file_date"].max()
    else:
        result = pd.DataFrame()
        new_bookmark = last_date

    logger.info("Extracted %d rows from %d files (new bookmark: %s)",
                len(result), len(frames), new_bookmark)

    return result, new_bookmark

In [ ]:
def extract_api_products(config):
    """
    Extract products from the marketplace API JSON file.
    Returns a DataFrame.
    """
    filepath = config["api_file"]
    logger.info("Extracting API products from %s", filepath)

    with open(filepath, "r") as f:
        response = json.load(f)

    df = pd.DataFrame(response["products"])
    df["api_timestamp"] = response["timestamp"]

    logger.info("Extracted %d products from API", len(df))
    return df

## The Transform stage

In [ ]:
def transform_orders(df):
    """
    Transform raw orders.
    - Parse order_date to datetime
    - Standardise currency to uppercase
    - Add extraction timestamp
    """
    logger.info("Transforming %d orders", len(df))

    result = df.copy()
    result["order_date"] = pd.to_datetime(result["order_date"], errors="coerce")
    result["currency"] = result["currency"].str.upper()
    result["loaded_at"] = datetime.now().isoformat()

    # Data quality check
    null_dates = result["order_date"].isna().sum()
    if null_dates > 0:
        logger.warning("%d orders have unparseable dates", null_dates)

    logger.info("Transform complete: %d rows", len(result))
    return result

In [ ]:
def transform_seller_products(df, exchange_rates):
    """
    Transform seller product data.
    - Clean strings, normalise names
    - Replace non-breaking spaces
    - Convert prices to GBP
    - Deduplicate
    """
    if len(df) == 0:
        logger.info("No seller data to transform")
        return df

    logger.info("Transforming %d seller products", len(df))

    result = df.copy()

    # String cleaning
    result["product_name"] = (
        result["product_name"]
        .str.replace("\xa0", " ", regex=False)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.title()
    )

    result["currency"] = result["currency"].str.upper()
    result["category"] = result["category"].str.strip().str.lower()

    # Price conversion
    result["price"] = pd.to_numeric(result["price"], errors="coerce")
    rate = result["currency"].map(exchange_rates)
    result["price_gbp"] = (result["price"] * rate).round(2)

    # Deduplicate: keep latest version of each product per seller
    before_dedup = len(result)
    result = result.drop_duplicates(
        subset=["product_id", "seller_name"],
        keep="last"
    )
    dupes_removed = before_dedup - len(result)

    if dupes_removed > 0:
        logger.info("Removed %d duplicates", dupes_removed)

    result["loaded_at"] = datetime.now().isoformat()

    logger.info("Transform complete: %d rows", len(result))
    return result.reset_index(drop=True)

## The Load stage

In [ ]:
def create_destination_tables(engine):
    """
    Create destination tables if they do not exist.
    """
    with engine.connect() as conn:
        conn.execute(text("""
            CREATE TABLE IF NOT EXISTS analytics_orders (
                order_id INTEGER PRIMARY KEY,
                customer_id TEXT,
                seller_id TEXT,
                order_date TEXT,
                total_amount REAL,
                currency TEXT,
                status TEXT,
                loaded_at TEXT
            )
        """))
        conn.execute(text("""
            CREATE TABLE IF NOT EXISTS analytics_products (
                product_id TEXT,
                product_name TEXT,
                price REAL,
                currency TEXT,
                price_gbp REAL,
                category TEXT,
                seller_name TEXT,
                file_date TEXT,
                loaded_at TEXT,
                PRIMARY KEY (product_id, seller_name)
            )
        """))
        conn.commit()
    logger.info("Destination tables ready")

In [ ]:
def load_orders(engine, df):
    """
    Load orders into the analytics database using upsert.
    """
    if len(df) == 0:
        logger.info("No orders to load")
        return

    logger.info("Loading %d orders", len(df))

    with engine.begin() as conn:
        for _, row in df.iterrows():
            conn.execute(text("""
                INSERT OR REPLACE INTO analytics_orders
                (order_id, customer_id, seller_id, order_date, total_amount, currency, status, loaded_at)
                VALUES (:order_id, :customer_id, :seller_id, :order_date, :total_amount, :currency, :status, :loaded_at)
            """), {
                "order_id": int(row["order_id"]),
                "customer_id": str(row["customer_id"]),
                "seller_id": str(row["seller_id"]),
                "order_date": str(row["order_date"]),
                "total_amount": float(row["total_amount"]),
                "currency": str(row["currency"]),
                "status": str(row["status"]),
                "loaded_at": str(row["loaded_at"]),
            })

    with engine.connect() as conn:
        count = conn.execute(text("SELECT COUNT(*) FROM analytics_orders")).fetchone()[0]
    logger.info("Orders table now has %d rows", count)

In [ ]:
def load_seller_products(engine, df):
    """
    Load seller products into the analytics database using upsert.
    """
    if len(df) == 0:
        logger.info("No seller products to load")
        return

    logger.info("Loading %d seller products", len(df))

    cols = ["product_id", "product_name", "price", "currency",
            "price_gbp", "category", "seller_name", "file_date", "loaded_at"]

    with engine.begin() as conn:
        for _, row in df.iterrows():
            params = {}
            for c in cols:
                val = row.get(c)
                if pd.isna(val):
                    params[c] = None
                elif isinstance(val, float):
                    params[c] = float(val)
                else:
                    params[c] = str(val)

            conn.execute(text("""
                INSERT OR REPLACE INTO analytics_products
                (product_id, product_name, price, currency, price_gbp, category, seller_name, file_date, loaded_at)
                VALUES (:product_id, :product_name, :price, :currency, :price_gbp, :category, :seller_name, :file_date, :loaded_at)
            """), params)

    with engine.connect() as conn:
        count = conn.execute(text("SELECT COUNT(*) FROM analytics_products")).fetchone()[0]
    logger.info("Products table now has %d rows", count)

## Wiring it together

Now the pipeline itself. This is the function you would schedule to run daily.

In [ ]:
def run_pipeline(config):
    """
    Run the complete Bazaar ETL pipeline.

    Returns an updated config dict with new bookmarks.
    """
    logger.info("=" * 60)
    logger.info("PIPELINE START")
    logger.info("=" * 60)

    # --- Destination setup ---
    dest_engine = create_engine(config["dest_db"])
    create_destination_tables(dest_engine)

    # --- EXTRACT ---
    logger.info("-" * 40)
    logger.info("STAGE: EXTRACT")
    logger.info("-" * 40)

    orders_raw, new_order_bookmark = extract_orders(config)
    seller_raw, new_seller_bookmark = extract_seller_files(config)
    # api_raw = extract_api_products(config)  # Available but not used in this pipeline run

    # --- TRANSFORM ---
    logger.info("-" * 40)
    logger.info("STAGE: TRANSFORM")
    logger.info("-" * 40)

    orders_clean = transform_orders(orders_raw)
    seller_clean = transform_seller_products(seller_raw, config["exchange_rates"])

    # --- LOAD ---
    logger.info("-" * 40)
    logger.info("STAGE: LOAD")
    logger.info("-" * 40)

    load_orders(dest_engine, orders_clean)
    load_seller_products(dest_engine, seller_clean)

    # --- Update bookmarks ---
    updated_config = config.copy()
    updated_config["last_order_id"] = new_order_bookmark
    updated_config["last_seller_file_date"] = new_seller_bookmark

    logger.info("=" * 60)
    logger.info("PIPELINE COMPLETE")
    logger.info("=" * 60)

    return updated_config

In [ ]:
# Run the pipeline
updated_config = run_pipeline(CONFIG)

Read the logs from top to bottom. You can see exactly what happened at each stage: how many rows were extracted, transformed, and loaded. If something went wrong, you would see the warning or error in context.

Now let's verify the output.

In [ ]:
# Check the destination database
dest_engine = create_engine(updated_config["dest_db"])

with dest_engine.connect() as conn:
    orders_count = conn.execute(text("SELECT COUNT(*) FROM analytics_orders")).fetchone()[0]
    products_count = conn.execute(text("SELECT COUNT(*) FROM analytics_products")).fetchone()[0]

print(f"Analytics orders:   {orders_count}")
print(f"Analytics products: {products_count}")

In [ ]:
# Sample some loaded orders
with dest_engine.connect() as conn:
    sample = pd.read_sql("SELECT * FROM analytics_orders LIMIT 5", conn)
sample

In [ ]:
# Sample some loaded products
with dest_engine.connect() as conn:
    sample = pd.read_sql("SELECT * FROM analytics_products LIMIT 5", conn)
sample

## Idempotency: run it again

The real test of a pipeline is what happens when you run it a second time. With our bookmarks in place, the second run should extract zero new rows and leave the tables unchanged.

In [ ]:
# Run the pipeline again with the updated bookmarks
final_config = run_pipeline(updated_config)

In [ ]:
# Verify: same counts as before
with dest_engine.connect() as conn:
    orders_count_2 = conn.execute(text("SELECT COUNT(*) FROM analytics_orders")).fetchone()[0]
    products_count_2 = conn.execute(text("SELECT COUNT(*) FROM analytics_products")).fetchone()[0]

print(f"Orders after 2nd run:   {orders_count_2} (was {orders_count})")
print(f"Products after 2nd run: {products_count_2} (was {products_count})")

Zero new orders extracted. Zero new seller products. The bookmarks worked. The pipeline is idempotent.

This is the behaviour you want. If a scheduler runs the pipeline twice by accident, or if you need to rerun after investigating an issue, the result is the same. No duplicates. No corrupted data.

## Incremental vs full refresh

Our pipeline uses incremental extraction (bookmarks). This is efficient for large datasets that grow over time, like orders and product updates.

But some data is better handled with a full refresh. Small reference tables -- sellers, categories, exchange rates -- change rarely and are small enough to reload completely. There is no point tracking bookmarks for a 20-row sellers table.

A common pattern is to use both in the same pipeline:

```python
# Incremental: large, append-only data
orders = extract_orders(config)          # Uses bookmark
load_orders(orders)                       # Upsert

# Full refresh: small reference data
sellers = extract_all_sellers(config)     # No bookmark
load_sellers(sellers)                     # Replace
```

Choose the strategy that fits each data source. There is no single right answer.

## Error handling

What happens when things go wrong? In a production pipeline, you need to think about three failure modes:

1. **Extract fails** -- the source is unavailable, the query times out, the file is missing
2. **Transform fails** -- unexpected data types, schema changes, corrupt data
3. **Load fails** -- the destination is unavailable, disk full, constraint violations

The golden rule: **fail loudly and leave things in a known state**.

In [ ]:
def run_pipeline_safe(config):
    """
    Run the pipeline with error handling at each stage.
    """
    logger.info("=" * 60)
    logger.info("PIPELINE START (safe mode)")
    logger.info("=" * 60)

    dest_engine = create_engine(config["dest_db"])
    create_destination_tables(dest_engine)

    updated_config = config.copy()

    # --- Orders pipeline ---
    try:
        logger.info("--- Orders pipeline ---")
        orders_raw, new_bookmark = extract_orders(config)
        orders_clean = transform_orders(orders_raw)
        load_orders(dest_engine, orders_clean)
        updated_config["last_order_id"] = new_bookmark
        logger.info("Orders pipeline: SUCCESS")
    except Exception as e:
        logger.error("Orders pipeline FAILED: %s", e)
        # Do NOT update the bookmark -- next run will retry from the same point

    # --- Seller products pipeline ---
    try:
        logger.info("--- Seller products pipeline ---")
        seller_raw, new_bookmark = extract_seller_files(config)
        seller_clean = transform_seller_products(seller_raw, config["exchange_rates"])
        load_seller_products(dest_engine, seller_clean)
        updated_config["last_seller_file_date"] = new_bookmark
        logger.info("Seller products pipeline: SUCCESS")
    except Exception as e:
        logger.error("Seller products pipeline FAILED: %s", e)

    logger.info("=" * 60)
    logger.info("PIPELINE COMPLETE")
    logger.info("=" * 60)

    return updated_config

Notice two important design decisions:

1. **Each data source has its own try/except block.** If the orders pipeline fails, the seller pipeline still runs. One failure does not cascade.

2. **Bookmarks are only updated on success.** If extraction fails, the bookmark stays at the old value. On the next run, the pipeline will retry from the same point. This is how you get automatic retry behaviour.

In a real system, you would also want:
- Alerting (send a Slack message or email on failure)
- Retry logic with exponential backoff
- Dead letter queues for data that repeatedly fails to load

But the core pattern is the same: catch errors at stage boundaries, log them, and decide whether to continue or stop.

In [ ]:
# Test the safe pipeline
safe_config = run_pipeline_safe(CONFIG)

## The complete picture

Let's zoom out and look at what we have built across all four notebooks:

```
Sources                    Pipeline                    Destination
--------                   --------                    -----------
SQLite DB    --extract-->  DataFrame  --transform-->   DataFrame  --load-->  Analytics DB
CSV files    --extract-->  DataFrame  --transform-->   DataFrame  --load-->  Analytics DB
API JSON     --extract-->  DataFrame  --transform-->   DataFrame  --load-->  Analytics DB
```

Each arrow is a function with a clear contract. The data flows through DataFrames, which are inspectable at every stage. Bookmarks ensure idempotency. Transactions ensure consistency. Logging ensures visibility.

This is not a toy example. This is the architecture that production data pipelines at companies of all sizes are built on. The specific tools change (Airflow instead of a script, Snowflake instead of SQLite, Kafka instead of CSV files), but the patterns are the same.

## Summary

What we covered:

- **The pipeline contract** -- `extract() -> DataFrame`, `transform(DataFrame) -> DataFrame`, `load(DataFrame) -> None`
- **Configuration dicts** -- parameterise the pipeline, keep credentials and paths out of the code
- **Logging** -- log at every stage boundary with row counts and status
- **Idempotency** -- run the pipeline twice, get the same result. Bookmarks make this possible.
- **Incremental vs full refresh** -- choose per data source based on size and update patterns
- **Error handling** -- catch errors at stage boundaries, do not update bookmarks on failure
- **Composability** -- independent functions that can be tested, replaced, and composed

---

## Exercises

### Exercise 1: Add the API source

Extend `run_pipeline` to also extract, transform, and load data from the marketplace API JSON file. You will need:

- A `transform_api_products(df, exchange_rates)` function
- A `load_api_products(engine, df)` function
- A new destination table (`analytics_api_products`)

Run the full pipeline and verify all three destination tables are populated.

In [ ]:
# Your code here


### Exercise 2: Bookmark persistence

Our bookmarks are stored in a Python dict, which is lost when the notebook restarts. Write two functions:

- `save_bookmarks(config, filepath)` -- save the bookmark values to a JSON file
- `load_bookmarks(config, filepath)` -- load bookmarks from the JSON file and merge them into the config

Modify the pipeline to save bookmarks after a successful run and load them at startup.

In [ ]:
# Your code here


### Exercise 3: Pipeline metrics

Write a version of the pipeline that collects and returns metrics for each stage:

```python
{
    "run_start": "2024-03-15T10:30:00",
    "run_end": "2024-03-15T10:30:05",
    "stages": {
        "extract_orders": {"rows": 2000, "status": "success"},
        "transform_orders": {"rows_in": 2000, "rows_out": 2000, "status": "success"},
        "load_orders": {"rows_loaded": 2000, "status": "success"},
        ...
    }
}
```

This is the kind of metadata that monitoring dashboards consume.

In [ ]:
# Your code here


### Exercise 4: Build a new pipeline

Build a complete ETL pipeline for the Bazaar **customers** data:

1. Extract: Pull all customers from the source SQLite database
2. Transform: Standardise country names to uppercase, validate email format (must contain `@`), add a `loaded_at` timestamp
3. Load: Upsert into an `analytics_customers` table in the destination database

Include logging, a transaction in the load step, and row count verification.

Run it twice to verify idempotency.

In [ ]:
# Your code here
